# 01 — Extract walking clips from AMASS CMU

This notebook finds **forward walking** in the AMASS version of the CMU mocap data and saves it as clean segments.

It uses two filters:

1. **Text filter.** The official CMU clip descriptions (e.g. `07_04  slow walk`) sort each clip into
   walk, run, mixed, or excluded (backwards, sideways, stairs, crawling…). Turning walks and stylised walks (sad, drunk, zombie…) get their own flags.
2. **Motion filter.** Each candidate clip is scanned in sliding windows. A window counts as walking when the root
   moves **forward** (along the direction the body faces) at walking speed, on a fairly straight path, with a steady
   pelvis height and the left and right hips swinging in **opposite phase**. Walking windows are merged into segments.

No SMPL body model is needed for this: everything comes from `trans`, the global orientation and the two hip rotations.

**Outputs** (in the `walking` folder inside your CMU folder):
- `clip_labels.csv`: every AMASS CMU file with its description and text category
- `walking_segments.csv`: one row per walking segment, with forward speed, cadence and stride length
- `segments/*.npz`: each segment with all original AMASS arrays cut to its frames (same keys as AMASS, plus metadata)
- `clips/<subject>/*.npz`: full copies of the original files that contain walking

## 1. Settings
Edit the paths and choices here. Everything else runs as is.

In [ ]:
from pathlib import Path

def find_project_root(start=Path.cwd()):
    for p in [start, *start.parents]:
        if (p / "pyproject.toml").exists():
            return p
    return start

PROJECT_ROOT = find_project_root()

# Folder that contains the AMASS CMU subject folders (01/, 02/, ...)
AMASS_CMU_DIR = Path(r"C:\Users\hasna\Documents\Good Practise\CMU")
OUT_DIR       = AMASS_CMU_DIR / "walking"                  # created automatically
INDEX_FILE    = OUT_DIR / "cmu-mocap-index-text.txt"       # downloaded automatically if missing
INDEX_URL     = "https://raw.githubusercontent.com/sfd158/cmu-mocap/master/cmu-mocap-index-text.txt"

# Which text categories to scan with the motion filter
INCLUDE_TURNING  = True   # 'walk, veer left', 'figure 8', 'walk around'... (only straight-enough windows are kept)
INCLUDE_STYLIZED = True   # 'sad walk', 'drunk walk', 'zombie walk', 'limp'... (flagged in the CSV so you can drop them later)
INCLUDE_MIXED    = True   # clips with several activities (e.g. 'walk, shake hands'); only the walking parts survive
INCLUDE_RUN      = False  # run/jog clips; if True, also raise SPEED_RANGE

# Motion filter thresholds
UP_AXIS          = 2            # AMASS is z-up
WIN_SEC          = 3.0          # window length for the walking test
HOP_SEC          = 0.5
SPEED_RANGE      = (0.3, 2.5)   # m/s, forward speed of a walking window
MIN_FWD_RATIO    = 0.8          # forward speed / total horizontal speed (rejects backwards and sideways)
MIN_STRAIGHTNESS = 0.7          # net displacement / path length inside a window
MAX_HEIGHT_STD   = 0.05         # m, pelvis height variation inside a window (rejects crouch/jump/stairs)
MAX_HIP_CORR     = -0.2         # left/right hip flexion must move in opposite phase
MIN_HIP_AMP_DEG  = 5.0          # std of (left - right) hip flexion, degrees
MIN_SEG_SEC      = 2.0          # shortest segment to keep
SMOOTH_SEC       = 0.25         # smoothing of the root path before taking velocities

SAVE_SEGMENT_NPZ = True    # walking frames only  -> OUT_DIR/segments/<clip>_segK.npz
COPY_FULL_CLIPS  = True    # whole original files that contain walking -> OUT_DIR/clips/<subject>/<file>.npz
print("project root:", PROJECT_ROOT)
print("AMASS CMU dir:", AMASS_CMU_DIR, "exists:", AMASS_CMU_DIR.exists())

In [ ]:
import re, json, urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.signal import savgol_filter
from scipy.spatial.transform import Rotation
from tqdm.auto import tqdm

assert AMASS_CMU_DIR.exists(), f"Folder not found: {AMASS_CMU_DIR}"
OUT_DIR.mkdir(parents=True, exist_ok=True)
(OUT_DIR / "segments").mkdir(exist_ok=True)
import shutil
pd.set_option("display.max_colwidth", 80)

## 2. CMU clip descriptions
Downloads the official description list (compiled by B. Hahne from CMU's index) once and labels every clip.

In [ ]:
if not INDEX_FILE.exists():
    INDEX_FILE.parent.mkdir(parents=True, exist_ok=True)
    print("downloading", INDEX_URL)
    urllib.request.urlretrieve(INDEX_URL, INDEX_FILE)

rows = []
for line in INDEX_FILE.read_text(encoding="utf-8", errors="replace").splitlines():
    m = re.match(r"^(\d+)_(\d+)\t(.*)$", line)
    if m:
        rows.append((f"{int(m[1]):02d}_{int(m[2]):02d}", int(m[1]), m[3].strip()))
desc = pd.DataFrame(rows, columns=["clip_id", "subject", "description"])
print(len(desc), "described clips")
desc.head()

In [ ]:
RX = lambda p: re.compile(p, re.I)
WALK      = RX(r"walk|stride|stroll|march|wander|hobble|limp")
RUN       = RX(r"\brun\b|run/|jog")
EXCLUDE = {
    "backward": RX(r"backward|bwk"),
    "sideways": RX(r"sideway|sidestep|crossover|cross legs|foot to foot"),
    "stairs_terrain": RX(r"stair|ladder|step(?:ping)? up|step over|up and over|uneven terrain|steop"),
    "not_gait": RX(r"moonwalk|crawl|walk jump|balancing|walk to run|run to sneak"),
}
MIXED     = RX(r"vignette|playground|;|pick up|carry|shake hands|catch|five|squat|\bsit|punch|attack|drink|"
               r"stretch(?!\s*walk)|kick|jump|dribble|dance|wash|sweep|wipe|conversation|look|search|stop|start")
TURNING   = RX(r"turn|veer|figure|digital|8\b|zigzag|around|navigate|path|obstacle|right ?angle|diagonal|walk (?:left|right)|wander|random")
STYLIZED  = RX(r"drunk|zombie|mummy|frankenstein|sneak|creep|limp|hurt|wound|achey|hobble|stumble|spastic|chicken|"
               r"dinosaur|cat walk|duck|pigeon|old man|sexy|macho|cool|ghetto|lavish|sad|depress|happy|excited|anger|"
               r"shy|scared|stern|attitude|bouncy|stiff|wild|bent|crouch|knees|legs apart|gangly|graceful|strong man|"
               r"heavyset|exaggerated|swinging|stretch walk|careful|silent|stealth|mickey|march|soldier|jaunt|mope|"
               r"clumsy|leaning|toes|confident|ball walk|arms out|evasive|traffic|link arms|hold hands")
SLOW, FAST = RX(r"slow"), RX(r"fast|brisk|quick|hurr")

def categorize(d):
    walk, run = bool(WALK.search(d)), bool(RUN.search(d))
    if not (walk or run):
        return "other"
    for name, rx in EXCLUDE.items():
        if rx.search(d):
            return f"excluded:{name}"
    if not walk:
        return "run"
    if run or MIXED.search(d) or len(re.findall(r",", d)) >= 2:
        return "mixed"
    return "walk"

desc["text_category"] = desc.description.map(categorize)
desc["turning"]  = desc.description.str.contains(TURNING)
desc["stylized"] = desc.description.str.contains(STYLIZED)
desc["speed_label"] = np.select([desc.description.str.contains(SLOW), desc.description.str.contains(FAST)],
                                ["slow", "fast"], "unspecified")
desc.text_category.value_counts()

In [ ]:
# Have a look at what ended up where
for cat in ["walk", "mixed", "excluded:backward", "excluded:stairs_terrain"]:
    sub = desc[desc.text_category == cat]
    print(f"\n== {cat} ({len(sub)}) ==")
    print(sub.description.str.lower().value_counts().head(12).to_string())

## 3. Match the AMASS files to the descriptions

In [ ]:
files = []
for p in sorted(AMASS_CMU_DIR.rglob("*.npz")):
    if OUT_DIR.resolve() in p.resolve().parents:   # skip our own output folder
        continue
    m = re.match(r"^(\d+)_(\d+)", p.stem)
    if m:  # skips shape.npz etc.
        files.append((f"{int(m[1]):02d}_{int(m[2]):02d}", str(p.relative_to(AMASS_CMU_DIR)), str(p)))
amass = pd.DataFrame(files, columns=["clip_id", "rel_path", "path"])
print(len(amass), "AMASS CMU motion files found")
assert len(amass) > 0, f"No *.npz files under {AMASS_CMU_DIR}: check AMASS_CMU_DIR in the settings cell."

clips = amass.merge(desc, on="clip_id", how="left")
clips["text_category"] = clips.text_category.fillna("no_description")
clips[["turning", "stylized"]] = clips[["turning", "stylized"]].fillna(False).astype(bool)
clips["speed_label"] = clips.speed_label.fillna("unspecified")
print("files without a description:", (clips.text_category == "no_description").sum())

keep = clips.text_category.eq("walk")
keep &= INCLUDE_TURNING  | ~clips.turning
keep &= INCLUDE_STYLIZED | ~clips.stylized
keep |= INCLUDE_MIXED & clips.text_category.eq("mixed")
keep |= INCLUDE_RUN   & clips.text_category.eq("run")
clips["candidate"] = keep
clips.drop(columns="path").to_csv(OUT_DIR / "clip_labels.csv", index=False)
print(clips.candidate.sum(), "candidate clips go to the motion filter")
clips[clips.candidate].text_category.value_counts()

## 4. Motion features
Works with both AMASS body-data formats: SMPL+H (`poses`, `mocap_framerate`) and SMPL-X (`root_orient`, `pose_body`, `mocap_frame_rate`).

- **Facing direction** = the body's forward axis (SMPL +z) rotated by the global orientation, projected onto the floor.
- **Forward speed** = root velocity · facing direction. This is the quantity your project measures.
- **Hip flexion** = rotation of the left/right hip about its local x-axis.

In [ ]:
def load_amass(path):
    d = np.load(path, allow_pickle=True)
    fps = float(d["mocap_framerate"] if "mocap_framerate" in d else d["mocap_frame_rate"])
    trans = d["trans"].astype(np.float64)
    if "poses" in d:
        go, hip_l, hip_r = d["poses"][:, 0:3], d["poses"][:, 3:6], d["poses"][:, 6:9]
    else:
        go, hip_l, hip_r = d["root_orient"], d["pose_body"][:, 0:3], d["pose_body"][:, 3:6]
    return dict(fps=fps, trans=trans, go=go.astype(np.float64),
                hip_l=hip_l.astype(np.float64), hip_r=hip_r.astype(np.float64))

HORIZ = [a for a in range(3) if a != UP_AXIS]
FACING_SIGN = 1.0   # auto-checked in the next cell

def frame_features(m):
    fps, T = m["fps"], len(m["trans"])
    w = max(5, int(SMOOTH_SEC * fps) // 2 * 2 + 1)
    pos = savgol_filter(m["trans"], min(w, T - (1 - T % 2)), 2, axis=0) if T > 7 else m["trans"]
    vel = np.gradient(pos, axis=0) * fps
    vh = vel[:, HORIZ]
    f = Rotation.from_rotvec(m["go"]).apply([0.0, 0.0, 1.0])[:, HORIZ] * FACING_SIGN
    f /= np.linalg.norm(f, axis=1, keepdims=True) + 1e-9
    flex = lambda rv: Rotation.from_rotvec(rv).as_euler("XYZ")[:, 0]
    return dict(fps=fps, pos=pos, fwd=(vh * f).sum(1), speed=np.linalg.norm(vh, axis=1),
                height=pos[:, UP_AXIS], hip_l=flex(m["hip_l"]), hip_r=flex(m["hip_r"]))

In [ ]:
# Sanity check of the facing direction on plain 'walk' clips: forward speed should be positive.
probe = clips[(clips.text_category == "walk") & ~clips.turning & ~clips.stylized].head(15)
ratios = []
for p in probe.path:
    F = frame_features(load_amass(p))
    ratios.append(F["fwd"].mean() / (F["speed"].mean() + 1e-9))
print("forward/total speed on plain walk clips:", np.round(ratios, 2))
if len(ratios) and np.median(ratios) < -0.5:
    FACING_SIGN = -1.0
    print("Facing axis points backwards in this data: flipped FACING_SIGN to -1.")
elif len(ratios) and abs(np.median(ratios)) < 0.5:
    print("WARNING: walking is mostly sideways to the computed facing direction. Check UP_AXIS / the facing axis.")

## 5. Find walking windows and merge them into segments

In [ ]:
def gait_frequency(l, r, fps, fmin=0.4, fmax=1.6):
    # Stride frequency (Hz) from the autocorrelation of left-minus-right hip flexion.
    s = (l - r) - (l - r).mean()
    ac = np.correlate(s, s, "full")[len(s) - 1:]
    ac /= ac[0] + 1e-12
    lo, hi = int(fps / fmax), min(int(fps / fmin), len(ac) - 1)
    if hi <= lo:
        return np.nan
    k = lo + int(np.argmax(ac[lo:hi]))
    return fps / k if ac[k] > 0.2 else np.nan

def walking_windows(F):
    fps, T = F["fps"], len(F["fwd"])
    W, H = int(WIN_SEC * fps), max(1, int(HOP_SEC * fps))
    out = []
    for s in range(0, max(T - W, 0) + 1, H):
        e = s + W
        if e > T:
            break
        fwd, spd = F["fwd"][s:e].mean(), F["speed"][s:e].mean()
        p = F["pos"][s:e][:, HORIZ]
        path = np.linalg.norm(np.diff(p, axis=0), axis=1).sum()
        l, r = F["hip_l"][s:e], F["hip_r"][s:e]
        out.append(dict(
            start=s, end=e, fwd_speed=fwd,
            fwd_ratio=fwd / (spd + 1e-9),
            straightness=np.linalg.norm(p[-1] - p[0]) / (path + 1e-9),
            height_std=F["height"][s:e].std(),
            hip_corr=np.corrcoef(l, r)[0, 1] if l.std() > 1e-6 and r.std() > 1e-6 else 0.0,
            hip_amp_deg=np.degrees((l - r).std()),
        ))
    w = pd.DataFrame(out)
    if len(w):
        w["is_walk"] = (w.fwd_speed.between(*SPEED_RANGE) & (w.fwd_ratio >= MIN_FWD_RATIO)
                        & (w.straightness >= MIN_STRAIGHTNESS) & (w.height_std <= MAX_HEIGHT_STD)
                        & (w.hip_corr <= MAX_HIP_CORR) & (w.hip_amp_deg >= MIN_HIP_AMP_DEG))
    return w

def segments_from_windows(w, F):
    fps, T = F["fps"], len(F["fwd"])
    mask = np.zeros(T, bool)
    for s, e in w.loc[w.is_walk, ["start", "end"]].itertuples(index=False):
        mask[s:e] = True
    # trim edges where the person is not (yet / any more) moving forward at walking speed
    k = max(1, int(0.5 * fps))
    moving = np.convolve(F["fwd"], np.ones(k) / k, mode="same") >= SPEED_RANGE[0]
    segs = []
    edges = np.flatnonzero(np.diff(np.r_[0, mask.astype(int), 0]))
    for s, e in zip(edges[::2], edges[1::2]):
        idx = np.flatnonzero(moving[s:e])
        if len(idx):
            s, e = s + idx[0], s + idx[-1] + 1
            if (e - s) >= MIN_SEG_SEC * fps:
                segs.append((int(s), int(e)))
    return segs

In [ ]:
seg_rows, failed = [], []
for row in tqdm(clips[clips.candidate].itertuples(), total=int(clips.candidate.sum())):
    try:
        m = load_amass(row.path)
        F = frame_features(m)
        w = walking_windows(F)
        if not len(w):
            continue
        for k, (s, e) in enumerate(segments_from_windows(w, F)):
            f_stride = gait_frequency(F["hip_l"][s:e], F["hip_r"][s:e], F["fps"])
            v = F["fwd"][s:e].mean()
            seg_id = f"{row.clip_id}_seg{k}"
            seg_rows.append(dict(
                segment_id=seg_id, clip_id=row.clip_id, subject=int(row.clip_id.split("_")[0]),
                rel_path=row.rel_path, description=row.description, text_category=row.text_category,
                turning=row.turning, stylized=row.stylized, speed_label=row.speed_label,
                fps=F["fps"], start_frame=s, end_frame=e, duration_s=(e - s) / F["fps"],
                fwd_speed_mps=v, fwd_speed_std=F["fwd"][s:e].std(),
                stride_freq_hz=f_stride, cadence_spm=2 * 60 * f_stride, stride_length_m=v / f_stride,
                clip_walk_fraction=(e - s) / len(F["fwd"]),
            ))
            if SAVE_SEGMENT_NPZ:
                d = np.load(row.path, allow_pickle=True)
                T = len(d["trans"])
                arrs = {k2: (d[k2][s:e] if d[k2].ndim and d[k2].shape[0] == T else d[k2]) for k2 in d.files}
                np.savez_compressed(OUT_DIR / "segments" / f"{seg_id}.npz", **arrs,
                                    source_file=row.rel_path, source_start=s, source_end=e,
                                    description=str(row.description))
    except Exception as ex:
        failed.append((row.rel_path, repr(ex)))

segs = pd.DataFrame(seg_rows)
segs.to_csv(OUT_DIR / "walking_segments.csv", index=False)
print(f"{len(segs)} walking segments from {segs.clip_id.nunique() if len(segs) else 0} clips, "
      f"{segs.duration_s.sum() / 60 if len(segs) else 0:.1f} minutes, {segs.subject.nunique() if len(segs) else 0} subjects")
if failed:
    print(len(failed), "files failed, e.g.:", failed[:3])
segs.head()

## 5b. Copy the full walking clips
Copies every original AMASS file that has at least one walking segment into `OUT_DIR/clips/`, keeping the subject folders. The originals are left untouched.

In [ ]:
if COPY_FULL_CLIPS and len(segs):
    clip_dir = OUT_DIR / "clips"
    copied = 0
    for rel in segs.rel_path.unique():
        dst = clip_dir / rel
        dst.parent.mkdir(parents=True, exist_ok=True)
        if not dst.exists():
            shutil.copy2(AMASS_CMU_DIR / rel, dst)
            copied += 1
    print(f"{segs.rel_path.nunique()} walking clips in {clip_dir} ({copied} newly copied)")

## 6. Summary

In [ ]:
if len(segs):
    print(segs.groupby("text_category").agg(segments=("segment_id", "size"), minutes=("duration_s", lambda x: x.sum() / 60)))
    print()
    print(segs.groupby("speed_label").fwd_speed_mps.describe()[["count", "mean", "min", "max"]].round(2))

    fig, ax = plt.subplots(1, 3, figsize=(15, 4))
    ax[0].hist(segs.fwd_speed_mps, bins=40, weights=segs.duration_s / 60)
    ax[0].set(xlabel="forward speed (m/s)", ylabel="minutes", title="Speed distribution")
    for lab, g in segs.groupby("speed_label"):
        ax[1].scatter(g.cadence_spm, g.stride_length_m, s=10, alpha=.6, label=lab)
    ax[1].set(xlabel="cadence (steps/min)", ylabel="stride length (m)", title="Cadence vs stride length"); ax[1].legend()
    order = [l for l in ["slow", "unspecified", "fast"] if l in set(segs.speed_label)]
    ax[2].boxplot([segs.loc[segs.speed_label == l, "fwd_speed_mps"] for l in order])
    ax[2].set_xticks(range(1, len(order) + 1), order)
    ax[2].set(ylabel="forward speed (m/s)", title="Measured speed vs text label")
    plt.tight_layout(); plt.show()

Check: walks labelled *slow* should come out slower than *fast* ones, and speed should be about cadence × stride length / 120.

In [ ]:
if len(segs):
    check = segs.dropna(subset=["cadence_spm"])
    pred = check.cadence_spm * check.stride_length_m / 120
    print("speed vs cadence*stride/120, max abs diff:", float((pred - check.fwd_speed_mps).abs().max()))
    print("segments without a clear gait frequency:", segs.cadence_spm.isna().sum())

## 7. Inspect one clip
Shows the root path from above, forward speed over time with walking segments shaded, and the two hip angles.

In [ ]:
def inspect(clip_id):
    row = clips[clips.clip_id == clip_id].iloc[0]
    F = frame_features(load_amass(row.path))
    w = walking_windows(F)
    t = np.arange(len(F["fwd"])) / F["fps"]
    fig, ax = plt.subplots(1, 3, figsize=(16, 4))
    p = F["pos"][:, HORIZ]
    ax[0].plot(p[:, 0], p[:, 1]); ax[0].plot(*p[0], "go"); ax[0].axis("equal")
    ax[0].set(title=f"{clip_id}: {row.description}", xlabel="m", ylabel="m")
    ax[1].plot(t, F["fwd"], label="forward"); ax[1].plot(t, F["speed"], alpha=.5, label="total horizontal")
    for s, e in segments_from_windows(w, F):
        ax[1].axvspan(s / F["fps"], e / F["fps"], color="g", alpha=.15)
    ax[1].set_ylim(min(0.0, F["fwd"].min()) - 0.1, F["speed"].max() + 0.3)
    ax[1].set(xlabel="s", ylabel="m/s", title="speed (green = kept)"); ax[1].legend()
    ax[2].plot(t, np.degrees(F["hip_l"]), label="left hip"); ax[2].plot(t, np.degrees(F["hip_r"]), label="right hip")
    ax[2].set(xlabel="s", ylabel="deg", title="hip flexion"); ax[2].legend()
    plt.tight_layout(); plt.show()

example = segs.clip_id.iloc[0] if len(segs) else clips.clip_id.iloc[0]
inspect(example)

In [ ]:
# Try others, e.g. a slow and a brisk walk:
# inspect("07_04"); inspect("07_12")

## Notes
- **Subject IDs:** CMU sometimes gives the same person several subject numbers. Keep that in mind for subject-wise splits.
- **Frame rate:** segments keep the original rate (mostly 120 Hz in CMU). Resample in the next preprocessing step.
- **Tuning:** if you lose too much, loosen `MIN_STRAIGHTNESS` or `MAX_HEIGHT_STD`; use `inspect()` to see why a clip was rejected.
- To build a strict "plain walking" subset later: `segs[(segs.text_category == "walk") & ~segs.stylized]`.